In [ ]:
# Standard Python imports
import os
import copy
import h5py
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Astropy imports
from astropy.io import fits
import astropy.io.fits as pyfits

# Lenstronomy imports
import lenstronomy
from lenstronomy.Data.psf import PSF
from lenstronomy.Util import mask_util, util

# this is the linear inversion. The kwargs will be updated afterwards
from lenstronomy.ImSim.image_linear_solve import ImageLinearFit
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.PointSource.point_source import PointSource
from lenstronomy.LensModel.lens_model import LensModel
import pickle
from lenstronomy.Analysis.light_profile import LightProfileAnalysis
from lenstronomy.Util import param_util
from lenstronomy.Sampling.parameters import Param
import corner


In [ ]:
filename = f'joint_modeling/{system_name}/{system_name}_joint.pkl'

class CustomUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Handle missing LikelihoodAddition class
        if name == 'LikelihoodAddition':
            class LikelihoodAddition:
                pass
            return LikelihoodAddition
        return super().find_class(module, name)

# load the saved data
with open(filename, "rb") as f:
    loaded_data = CustomUnpickler(f).load()

kwargs_result = loaded_data["kwargs_result"]
multi_band_list = loaded_data["multi_band_list"]
kwargs_model = loaded_data["kwargs_model"]
kwargs_params = loaded_data.get("kwargs_params", None)
chain_list = loaded_data.get('chain_list')
kwargs_constraints = loaded_data.get("kwargs_constraints", None)
kwargs_likelihood = loaded_data.get("kwargs_likelihood", None)
kwargs_data_joint = loaded_data.get("kwargs_data_joint", None)

def prep_phot_pars_array():
    '''Get the essential HST conversions from Header files'''
    filename = f'cutout_data/{system_name}/{filt}/{system_name}_{filt}_cutout.fits'
    with fits.open(filename) as hdul:
            header = hdul[0].header
    photzpt = header['photzpt']
    photflam = header['photflam']
    photplam = header['photplam']
    return [photflam, photzpt, photplam]

def get_stmag(electron_flux):
    '''Calculate standard magnitude using flux'''
    photflam, photzpt, photplam = prep_phot_pars_array()
    flux = np.asarray(electron_flux)
    flux = flux * photflam
    return -2.5 * np.log10(flux) + photzpt

def get_abmag(electron_flux):
    '''Convert standard magnitude to AB magnitude'''
    stmag = get_stmag(electron_flux)
    _, _, photplam = prep_phot_pars_array()
    return stmag - 5. * np.log10(photplam) + 2.5 * np.log10(299792458e10) - 27.5
    

In [ ]:
filters = ["F160W", "F814W", "F475X"]
band_map = {"F160W": 0, "F814W": 1, "F475X": 2}

sampler_type, samples_mcmc, param_mcmc, dist_mcmc = chain_list[4]

kwargs_params = loaded_data.get('kwargs_params')
kwargs_fixed_lens = kwargs_params['lens_model'][2]
kwargs_fixed_lens_light = kwargs_params['lens_light_model'][2]
kwargs_fixed_source = kwargs_params['source_model'][2]
kwargs_fixed_ps = kwargs_params['point_source_model'][2]

param = Param(
    kwargs_model,
    kwargs_fixed_lens=kwargs_fixed_lens,
    kwargs_fixed_lens_light=kwargs_fixed_lens_light,
    kwargs_fixed_source=kwargs_fixed_source,
    kwargs_fixed_ps=kwargs_fixed_ps,
    **kwargs_constraints
    )
    
ellipticity_results = {
    "F160W": {"phi": [], "q": [], "r_eff": []},
    "F814W": {"phi": [], "q": [], "r_eff": []},
    "F475X": {"phi": [], "q": [], "r_eff": []},
}

grid_spacing = 0.02 
grid_num = 200

uniform_index = kwargs_model['lens_light_model_list'].index('UNIFORM')

mcmc_flux_list = []

for i in range(len(samples_mcmc) - 50000, len(samples_mcmc)):

    kwargs_out = param.args2kwargs(samples_mcmc[i])

    kwargs_lens_all = kwargs_out['kwargs_lens']
    kwargs_lens_light_all = kwargs_out['kwargs_lens_light']
    kwargs_source_all = kwargs_out['kwargs_source']
    kwargs_ps_all = kwargs_out['kwargs_ps']

    fluxes_this_sample = []

    for filt in filters:

        i_band = band_map[filt]

        kwargs_data = multi_band_list[i_band][0]
        kwargs_psf = multi_band_list[i_band][1]
        kwargs_numerics = multi_band_list[i_band][2]
        likelihood_mask = kwargs_likelihood['image_likelihood_mask_list'][i_band]

        data_class = ImageData(**kwargs_data)
        psf_class = PSF(**kwargs_psf)

        # Select lens components per band 

        if filt == "F160W":
            base_indices = [0, 1]

        elif filt == "F814W":
            if system_name in ['J0602-4335', 'J1001+5027']:
                base_indices = [2, 3]
            else:
                base_indices = [2]

        elif filt == "F475X":
            if system_name in ['J0602-4335', 'J1001+5027']:
                base_indices = [4, 5]
            else:
                base_indices = [3]

        lens_indices = base_indices + [uniform_index]

        lens_light_model_list = [
            kwargs_model['lens_light_model_list'][k]
            for k in lens_indices
        ]

        lens_light_kwargs = [
            kwargs_lens_light_all[k]
            for k in lens_indices
        ]

        # source per band
        if system_name == "J2325-5229":
            if filt == "F160W":
                # include main source + SHAPELET in IR
                source_indices = [0, 1]
            else:
                source_indices = [0]
        else:
            # all other systems use default single source
            source_indices = [0]

        source_model_list = [
            kwargs_model['source_light_model_list'][k]
            for k in source_indices
        ]

        source_kwargs = [
            kwargs_source_all[k]
            for k in source_indices
        ]

        lightModel = LightModel(lens_light_model_list)
        sourceModel = LightModel(source_model_list)
        pointSource = PointSource(kwargs_model['point_source_model_list'])
        lensModel = LensModel(kwargs_model['lens_model_list'])

        imageLinearFit = ImageLinearFit(
            data_class=data_class,
            psf_class=psf_class,
            lens_model_class=lensModel,
            source_model_class=sourceModel,
            lens_light_model_class=lightModel,
            point_source_class=pointSource,
            kwargs_numerics=kwargs_numerics,
            likelihood_mask=likelihood_mask,
            psf_error_map_bool_list=[True]
        )

        # linear solver
        imageLinearFit.image_linear_solve(
            kwargs_lens=kwargs_lens_all,
            kwargs_source=source_kwargs,
            kwargs_lens_light=lens_light_kwargs,
            kwargs_ps=kwargs_ps_all
        )

        # image fluxes
        flux_im1 = kwargs_ps_all[0]['point_amp'][0]
        flux_im2 = kwargs_ps_all[0]['point_amp'][1]

        # lens flux (exclude UNIFORM)
        imageModel = imageLinearFit  # inherits ImageModel methods

        flux_lens = 0

        # use only first light component for J1001
        if system_name == 'J1001+5027':
            flux_lens += np.sum(
                imageModel.lens_surface_brightness(
                    lens_light_kwargs, k=0
                )
            )
        else:
            for k, model_name in enumerate(lens_light_model_list):
                if model_name != 'UNIFORM':
                    flux_lens += np.sum(
                        imageModel.lens_surface_brightness(
                            lens_light_kwargs, k=k
                        )
                    )

        # host flux
        flux_host_lensed = np.sum(
            imageModel.source_surface_brightness(
                source_kwargs,
                kwargs_lens_all,
                de_lensed=False
            )
        )

        flux_host_intrinsic = np.sum(
            imageModel.source_surface_brightness(
                source_kwargs,
                kwargs_lens_all,
                de_lensed=True
            )
        )

        fluxes_this_sample.extend([
            flux_im1,
            flux_im2,
            flux_lens,
            flux_host_lensed,
            flux_host_intrinsic
        ])

        # morphology (exclude UNIFORM)
        if system_name == 'J1001+5027':
            # only first component = main lens
            model_bool_list = [False] * len(lens_light_model_list)
            model_bool_list[0] = True
        else:
            # use all non-UNIFORM components
            model_bool_list = [
                model_name != 'UNIFORM'
                for model_name in lens_light_model_list
            ]

        light_analysis = LightProfileAnalysis(lightModel)

        cx = lens_light_kwargs[0]["center_x"]
        cy = lens_light_kwargs[0]["center_y"]

        e1, e2 = light_analysis.ellipticity(
            lens_light_kwargs,
            center_x=cx,
            center_y=cy,
            grid_spacing=grid_spacing,
            grid_num=grid_num,
            model_bool_list=model_bool_list,
            iterative=True
        )

        r_eff = light_analysis.half_light_radius(
            lens_light_kwargs,
            center_x=cx,
            center_y=cy,
            grid_spacing=grid_spacing,
            grid_num=grid_num,
            model_bool_list=model_bool_list,
        )

        phi, q = param_util.ellipticity2phi_q(e1, e2)
        phi = (phi * 180 / np.pi) % 180

        ellipticity_results[filt]["phi"].append(phi)
        ellipticity_results[filt]["q"].append(q)
        ellipticity_results[filt]["r_eff"].append(r_eff)

    mcmc_flux_list.append(fluxes_this_sample)

mcmc_flux_array = np.array(mcmc_flux_list)


In [ ]:
# Helper function for median and 1-sigma uncertainties 
def get_median_and_uncertainties(samples):
    median = np.median(samples)
    lower, upper = np.percentile(samples, [16, 84])
    return median, (upper - median, median - lower)

# Number of flux parameters per filter
n_flux_per_filt = 5

# Compute medians and uncertainties for every flux column
n_params = mcmc_flux_array.shape[1]
results_dict = {filt: {} for filt in filters}

for i, filt in enumerate(filters):
    start_idx = i * n_flux_per_filt
    end_idx = start_idx + n_flux_per_filt
    
    # Assign appropriate flux labels
    flux_labels = ["Image1", "Image2", "Lens", "Host_lensed", "Host_intrinsic"]

    # Load calibration parameters for this filter 
    filename = f'cutout_data/{system_name}/{filt}/{system_name}_{filt}_cutout.fits'
    with fits.open(filename) as hdul:
        header = hdul[0].header
    photzpt = header['photzpt']
    photflam = header['photflam']
    photplam = header['photplam']

    # Iterate over each flux parameter 
    for j, label in enumerate(flux_labels):
        samples = mcmc_flux_array[:, start_idx + j]

        # Flux statistics
        flux_median, (flux_sigma_plus, flux_sigma_minus) = get_median_and_uncertainties(samples)

        # Magnitude statistics
        abmag_samples = get_abmag(samples)
        mag_median, (mag_sigma_plus, mag_sigma_minus) = get_median_and_uncertainties(abmag_samples)

        results_dict[filt][label] = {
            "flux_median": flux_median,
            "flux_sigma_plus": flux_sigma_plus,
            "flux_sigma_minus": flux_sigma_minus,
            "mag_median": mag_median,
            "mag_sigma_plus": mag_sigma_plus,
            "mag_sigma_minus": mag_sigma_minus
        }

# Print results in clean format
for filt in filters:
    print(f"\n{'='*20} {filt} {'='*20}")
    for label, stats in results_dict[filt].items():
        print(
            f"{label:15s}: "
            f"Flux = {stats['flux_median']:.2f} "
            f"(+{stats['flux_sigma_plus']:.2f}, -{stats['flux_sigma_minus']:.2f}) | "
            f"AB Mag = {stats['mag_median']:.2f} "
            f"(+{stats['mag_sigma_plus']:.2f}, -{stats['mag_sigma_minus']:.2f})"
        )


In [ ]:

outname = f'joint_modeling/{system_name}/{system_name}_photometry.hdf5'
os.makedirs(os.path.dirname(outname), exist_ok=True)

with h5py.File(outname, 'w') as f:

    # Loop over filters
    for filt in filters:
        grp = f.create_group(filt)  # create a group for each filter

        # point source fluxes & mags
        for idx, label in enumerate(["Image1", "Image2"]):
            stats = results_dict[filt][label]
            grp.create_dataset(f'image{idx+1}_flux', data=stats["flux_median"])
            grp.create_dataset(
                f'image{idx+1}_flux_sigma',
                data=[stats["flux_sigma_plus"], stats["flux_sigma_minus"]]
            )
            grp.create_dataset(f'image{idx+1}_mag', data=stats["mag_median"])
            grp.create_dataset(
                f'image{idx+1}_mag_sigma',
                data=[stats["mag_sigma_plus"], stats["mag_sigma_minus"]]
            )

        # lens
        lens_stats = results_dict[filt]["Lens"]
        grp.create_dataset('lens_flux', data=lens_stats["flux_median"])
        grp.create_dataset(
            'lens_flux_sigma',
            data=[lens_stats["flux_sigma_plus"], lens_stats["flux_sigma_minus"]]
        )
        grp.create_dataset('lens_mag', data=lens_stats["mag_median"])
        grp.create_dataset(
            'lens_mag_sigma',
            data=[lens_stats["mag_sigma_plus"], lens_stats["mag_sigma_minus"]]
        )

        # source QSO
        for host_label, dataset_prefix in zip(
            ["Host_lensed", "Host_intrinsic"],
            ["host_lensed", "host_intrinsic"]
        ):
            stats = results_dict[filt][host_label]
            grp.create_dataset(f'{dataset_prefix}_flux', data=stats["flux_median"])
            grp.create_dataset(
                f'{dataset_prefix}_flux_sigma',
                data=[stats["flux_sigma_plus"], stats["flux_sigma_minus"]]
            )
            grp.create_dataset(f'{dataset_prefix}_mag', data=stats["mag_median"])
            grp.create_dataset(
                f'{dataset_prefix}_mag_sigma',
                data=[stats["mag_sigma_plus"], stats["mag_sigma_minus"]]
            )

        # ellipticity
        ellip_grp = grp.create_group("ellipticity")

        ellip_grp.create_dataset(
            "phi_samples",
            data=np.asarray(ellipticity_results[filt]["phi"])
        )
        ellip_grp.create_dataset(
            "q_samples",
            data=np.asarray(ellipticity_results[filt]["q"])
        )
        ellip_grp.create_dataset(
            "r_eff_samples",
            data=np.asarray(ellipticity_results[filt]["r_eff"])
        )

print(f"Saved photometry + ellipticity posteriors to {outname}")
